# 02 - Pre-processamento: Landmarks Faciais, Head Pose e Features de Atencao

Objetivo: substituir os scripts legados (`extractFrames.py`/`hog.py`, que extraem TODOS os frames e HOG bruto) por um pipeline leve e reprodutivel:

1. Amostragem de N frames uniformemente espacados por clipe (nao todos os frames).
2. Deteccao de landmarks faciais com **MediaPipe FaceMesh** (468 pontos + iris, `refine_landmarks=True`).
3. Estimativa de **head pose** (yaw/pitch/roll via solvePnP).
4. Features de atencao: **EAR** (Eye Aspect Ratio, proxy de piscadas/sonolencia) e **MAR** (Mouth Aspect Ratio, proxy de bocejo/fala).
5. Features de **iris/gaze** (posicao normalizada da iris dentro do contorno do olho, horizontal e vertical) - inspirado em Sugihdharma & Bachtiar (2023, SAE-CNN, DOI:10.1145/3626641.3626938), que obtiveram a maior acuracia entre os trabalhos revisados (Secao 3 do plano) usando **apenas** marcos oculares e gaze extraidos via OpenFace.
6. Fallback explicito para frames sem rosto detectado (inspirado no PriorNet, arXiv:2605.03615 - ver Secao 3 do plano).
7. Persistencia em `.parquet` por split, pronto para as Trilhas A (ML classico) e B (modelo temporal).

**Importante:** rodar primeiro a celula de PILOTO (subconjunto pequeno) antes do full-run, conforme o cronograma da Secao 9 do plano.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
from tqdm import tqdm

RANDOM_STATE = 42
N_FRAMES_PER_CLIP = 20  # amostragem uniforme; ajustar se sobrar/faltar tempo (ver Secao 6 do plano)

ROOT = Path.cwd().parent
LABELS_DIR = ROOT / "datasets" / "DAiSEE" / "Labels"
VIDEOS_DIR = ROOT / "datasets" / "DAiSEE" / "DataSet"
FEATURES_DIR = ROOT / "datasets" / "DAiSEE" / "features"
FEATURES_DIR.mkdir(exist_ok=True)

mp_face_mesh = mp.solutions.face_mesh

# Indices de landmarks relevantes (MediaPipe FaceMesh, 468 pontos)
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]
MOUTH = [61, 291, 39, 181, 0, 17, 269, 405]
NOSE_TIP = 1
CHIN = 152
LEFT_EYE_CORNER = 33
RIGHT_EYE_CORNER = 263
LEFT_MOUTH = 61
RIGHT_MOUTH = 291

# Centros da iris (disponiveis apenas com refine_landmarks=True) - usados para features de gaze,
# na linha do SAE-CNN (Sugihdharma & Bachtiar, 2023): marcos oculares + gaze via OpenFace.
LEFT_IRIS_CENTER = 473
RIGHT_IRIS_CENTER = 468

In [ ]:
def eye_aspect_ratio(landmarks, idxs):
    p = np.array([[landmarks[i].x, landmarks[i].y] for i in idxs])
    a = np.linalg.norm(p[1] - p[5])
    b = np.linalg.norm(p[2] - p[4])
    c = np.linalg.norm(p[0] - p[3])
    return (a + b) / (2.0 * c + 1e-6)

def mouth_aspect_ratio(landmarks, idxs):
    p = np.array([[landmarks[i].x, landmarks[i].y] for i in idxs])
    vertical = np.linalg.norm(p[2] - p[3]) + np.linalg.norm(p[4] - p[5])
    horizontal = np.linalg.norm(p[0] - p[1])
    return vertical / (2.0 * horizontal + 1e-6)

def iris_gaze_ratio(landmarks, eye_idxs, iris_idx):
    """Posicao normalizada (0-1) da iris dentro do contorno do olho (proxy de gaze horizontal/vertical).
    eye_idxs segue a mesma convencao de eye_aspect_ratio: [p0,p1,p2,p3,p4,p5] com p0/p3 = cantos horizontais.
    """
    p = np.array([[landmarks[i].x, landmarks[i].y] for i in eye_idxs])
    iris = np.array([landmarks[iris_idx].x, landmarks[iris_idx].y])
    x_min, x_max = sorted([p[0][0], p[3][0]])
    y_min = min(p[1][1], p[2][1])
    y_max = max(p[4][1], p[5][1])
    gaze_h = (iris[0] - x_min) / (x_max - x_min + 1e-6)
    gaze_v = (iris[1] - y_min) / (y_max - y_min + 1e-6)
    return gaze_h, gaze_v

def estimate_head_pose(landmarks, img_w, img_h):
    # Modelo 3D generico de face (pontos canonicos aproximados) para solvePnP
    model_points = np.array([
        (0.0, 0.0, 0.0),          # nose tip
        (0.0, -63.6, -12.5),      # chin
        (-43.3, 32.7, -26.0),     # left eye corner
        (43.3, 32.7, -26.0),      # right eye corner
        (-28.9, -28.9, -24.1),    # left mouth corner
        (28.9, -28.9, -24.1),     # right mouth corner
    ], dtype=np.float64)

    idxs = [NOSE_TIP, CHIN, LEFT_EYE_CORNER, RIGHT_EYE_CORNER, LEFT_MOUTH, RIGHT_MOUTH]
    image_points = np.array([
        (landmarks[i].x * img_w, landmarks[i].y * img_h) for i in idxs
    ], dtype=np.float64)

    focal_length = img_w
    center = (img_w / 2, img_h / 2)
    camera_matrix = np.array([
        [focal_length, 0, center[0]],
        [0, focal_length, center[1]],
        [0, 0, 1],
    ], dtype=np.float64)
    dist_coeffs = np.zeros((4, 1))

    success, rvec, _ = cv2.solvePnP(model_points, image_points, camera_matrix, dist_coeffs)
    if not success:
        return np.nan, np.nan, np.nan

    rmat, _ = cv2.Rodrigues(rvec)
    sy = np.sqrt(rmat[0, 0] ** 2 + rmat[1, 0] ** 2)
    pitch = np.degrees(np.arctan2(-rmat[2, 0], sy))
    yaw = np.degrees(np.arctan2(rmat[1, 0], rmat[0, 0]))
    roll = np.degrees(np.arctan2(rmat[2, 1], rmat[2, 2]))
    return yaw, pitch, roll


In [ ]:
def sample_frame_indices(n_total_frames, n_samples):
    if n_total_frames <= 0:
        return []
    n = min(n_samples, n_total_frames)
    return np.linspace(0, n_total_frames - 1, num=n, dtype=int).tolist()

def extract_clip_features(video_path, n_samples=N_FRAMES_PER_CLIP):
    """Retorna uma lista de dicts (1 por frame amostrado) com landmarks/pose/EAR/MAR.
    Frames sem rosto detectado geram um placeholder (face_detected=False) - ver PriorNet, Secao 3 do plano.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []

    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = sample_frame_indices(n_total, n_samples)
    rows = []

    with mp_face_mesh.FaceMesh(
        static_image_mode=True, max_num_faces=1, refine_landmarks=True, min_detection_confidence=0.5
    ) as face_mesh:
        for frame_idx in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ok, frame = cap.read()
            if not ok:
                rows.append({"frame_idx": frame_idx, "face_detected": False})
                continue

            h, w = frame.shape[:2]
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result = face_mesh.process(rgb)

            if not result.multi_face_landmarks:
                rows.append({"frame_idx": frame_idx, "face_detected": False})
                continue

            lm = result.multi_face_landmarks[0].landmark
            ear = (eye_aspect_ratio(lm, LEFT_EYE) + eye_aspect_ratio(lm, RIGHT_EYE)) / 2.0
            mar = mouth_aspect_ratio(lm, MOUTH)
            yaw, pitch, roll = estimate_head_pose(lm, w, h)

            # Gaze (SAE-CNN, Sugihdharma & Bachtiar 2023): posicao da iris dentro do contorno do olho.
            gaze_h_left, gaze_v_left = iris_gaze_ratio(lm, LEFT_EYE, LEFT_IRIS_CENTER)
            gaze_h_right, gaze_v_right = iris_gaze_ratio(lm, RIGHT_EYE, RIGHT_IRIS_CENTER)
            gaze_h = (gaze_h_left + gaze_h_right) / 2.0
            gaze_v = (gaze_v_left + gaze_v_right) / 2.0
            gaze_offset = float(np.hypot(gaze_h - 0.5, gaze_v - 0.5))  # proxy de desvio do olhar do centro

            rows.append({
                "frame_idx": frame_idx,
                "face_detected": True,
                "ear": ear,
                "mar": mar,
                "yaw": yaw,
                "pitch": pitch,
                "roll": roll,
                "gaze_h": gaze_h,
                "gaze_v": gaze_v,
                "gaze_offset": gaze_offset,
            })

    cap.release()
    return rows

In [ ]:
def resolve_video_path(clip_id, split):
    user_id = clip_id[:6]
    stub = clip_id.replace(".avi", "")
    return VIDEOS_DIR / split / user_id / stub / clip_id

def build_feature_table(labels_csv, split_dir_name, limit=None):
    df = pd.read_csv(LABELS_DIR / labels_csv)
    df.columns = [c.strip() for c in df.columns]
    if limit:
        df = df.sample(n=min(limit, len(df)), random_state=RANDOM_STATE)

    all_rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=split_dir_name):
        clip_id = row["ClipID"]
        video_path = resolve_video_path(clip_id, split_dir_name)
        if not video_path.exists():
            continue
        frame_feats = extract_clip_features(video_path)
        for f in frame_feats:
            f["ClipID"] = clip_id
            f["Boredom"] = row["Boredom"]
            f["Engagement"] = row["Engagement"]
            f["Confusion"] = row["Confusion"]
            f["Frustration"] = row["Frustration"]
            all_rows.append(f)

    return pd.DataFrame(all_rows)

## PILOTO (rodar primeiro!): ~200 videos para validar o pipeline e medir tempo/clipe

In [ ]:
import time

t0 = time.time()
pilot_df = build_feature_table("TrainLabels.csv", "Train", limit=200)
elapsed = time.time() - t0
print(f"Piloto: {len(pilot_df)} linhas de frames em {elapsed:.1f}s")
print(f"Tempo medio por video: {elapsed/200:.2f}s -> projecao para dataset completo (~8925 videos): {elapsed/200*8925/60:.1f} min")
print("Taxa de deteccao de rosto no piloto:", pilot_df["face_detected"].mean() if len(pilot_df) else "N/A")
pilot_df.head()

## Full-run por split

**Decisao de fallback (ver Secao 9 do plano):** se a projecao de tempo do piloto acima for incompativel com o cronograma de 2 dias, reduzir via `limit=` (amostra estratificada) e documentar como limitacao explicita.

In [ ]:
# Ajustar `limit=None` para rodar o dataset completo, ou definir um valor caso o piloto indique risco de estouro de tempo
LIMIT = None  # ex.: 0.3 * n_clipes do split, ou None para processar tudo

train_feats = build_feature_table("TrainLabels.csv", "Train", limit=LIMIT)
train_feats.to_parquet(FEATURES_DIR / "train_frame_features.parquet", index=False)

val_feats = build_feature_table("ValidationLabels.csv", "Validation", limit=LIMIT)
val_feats.to_parquet(FEATURES_DIR / "validation_frame_features.parquet", index=False)

test_feats = build_feature_table("TestLabels.csv", "Test", limit=LIMIT)
test_feats.to_parquet(FEATURES_DIR / "test_frame_features.parquet", index=False)

print("Salvo em", FEATURES_DIR)

## Checklist de saida
- [ ] Piloto executado e taxa de deteccao de rosto (`face_detected`) documentada
- [ ] Projecao de tempo do full-run avaliada contra o cronograma (Secao 9 do plano)
- [ ] `train/validation/test_frame_features.parquet` gerados em `datasets/DAiSEE/features/`
- [ ] Se amostragem parcial foi usada (`LIMIT` != None), registrar isso como limitacao

Proximo passo: `03_baseline_classico.ipynb`.